## Read the data

In [254]:
import numpy as np
import pandas as pd
import sys
PATH_TO_PREDICTOR_HOME = "../.."
sys.path.append(PATH_TO_PREDICTOR_HOME)
METHOD = "hmm_pomegranate"
EXPERIMENT = 'simple_model_enrichment'


# 1. Read data (Now it is the same data)

In [255]:
DATA_PATH_BASE = 'G:/My Drive/Work/ITMO/data'
# DATA_PATH_BASE = 'D:/Google Drive/Work/ITMO/data' #windows
# DATA_PATH_BASE = '/home/denklewer/Data' #ubuntu

DATA_PATH = f'{DATA_PATH_BASE}/{EXPERIMENT}/per_length_per_kfold_split/' # work laptop
PATH_TO_SAVE_RESULTS = f'{DATA_PATH_BASE}/{EXPERIMENT}/peptide_cores/'
APPROACH = "DECOY_PATHS"

In [256]:
from data_reading_methods import get_available_alleles, read_data

ALLELES = list(sorted(get_available_alleles(DATA_PATH)))

per_allele_per_kfold_per_length_binders_train = read_data(DATA_PATH, ALLELES, "train", "binders")
per_allele_per_kfold_per_length_binders_test = read_data(DATA_PATH, ALLELES, "test", "binders")
# per_allele_per_kfold_per_length_non_binders_train = read_data(DATA_PATH, ALLELES, "train", "nonbinders")
# per_allele_per_kfold_per_length_non_binders_test = read_data(DATA_PATH, ALLELES, "test", "nonbinders")

target_class = "class2"


DataParams = {
"class1": {
    "ALLELES": sorted([ 'HLA-A*01:01', 'HLA-A*03:01']),
    "TARGET_LENGTHS":   [8, 9, 10, 11, 12],
},
    
"class2": {
    "ALLELES":sorted([
        #'HLA-DPA1*02:02/DPB1*05:01',
        'HLA-DQA1*05:01/DQB1*02:01']),
  #  "ALLELES":sorted(['HLA-DRB1*01:01']),
    "TARGET_LENGTHS":    [12, 13,14, 15, 16,17, 18],

    
    
}

}

ALLELES = DataParams[target_class]["ALLELES"]
TARGET_LENGTHS = DataParams[target_class]["TARGET_LENGTHS"]

selected_alleles = ALLELES
ALLELES




['HLA-DQA1*05:01/DQB1*02:01']

## 2.0 Merge alleles together

In [257]:
# combination_size = len(ALLELES)
# ALLELES = sorted(['HLA-A*01:01', 'HLA-A*03:01', 'HLA-A*02:01'])
# # ALLELES = sorted([ 'HLA-A*03:01'])
# ALLELES = sorted([   'HLA-DRB1*01:01',
#                     # 'HLA-DPA1*01:03/DPB1*04:01'
#                       ])
# ALLELES = sorted(['HLA-DPA1*01:03-DPB1*04:01'])
#
# ALLELES = sorted(['HLA-DRB1*01:01', 'HLA-DPA1*01:03-DPB1*04:01'])
#

combination_size = 1
TOTAL_SPLITS = 1
# TARGET_LENGTHS= [8, 9, 10, 11, 12]
RUNS = 5
USE_WEIGHTS = True

# SELECTED_RUN = 0
# SELECTED_RUN = {0,1,2,3,4,5, 6, 7, 8, 9}
SELECTED_RUN = {0,1}

# SELECTED_RUN = {0}
#
SELECTED_SPLIT = 0
models_to_read = ['v']

## Merge train data together (multiple alleles to single set of peptides)

In [258]:
from collections import defaultdict
global_selected_alleles = ALLELES # Data alleles we selected (but the number will be reduced by acceptable alleles)

selected_alleles = global_selected_alleles


per_allele_representations_train = dict()
per_allele_representations_test = dict()
per_name_representation_train = dict()
per_name_representation_test = dict()

def handle_intersections(per_allele_per_kfold_per_length_peptides):
    per_allele_result_peptide_sets =  defaultdict(set)
    per_allele_per_length_peptide_sets =  defaultdict(set)
    PER_LENGTH_SIZE = {target_length: min([len(set(per_allele_per_kfold_per_length_peptides[allele_name][0][target_length])) for allele_name in selected_alleles]) for target_length in TARGET_LENGTHS}
    print("Min unique sizes", PER_LENGTH_SIZE)
    for target_length in TARGET_LENGTHS:
        print(f"Length {target_length}")
        for allele_name in selected_alleles:
            if target_length in per_allele_per_kfold_per_length_peptides[allele_name][0]:
                print(f"initial length for  allele { allele_name} TL {target_length} is {len(per_allele_per_kfold_per_length_peptides[allele_name][0][target_length])}")
                print(f"unique length for  allele { allele_name} TL {target_length} is {len(set(per_allele_per_kfold_per_length_peptides[allele_name][0][target_length]))}")
                unique_peptides = np.unique(per_allele_per_kfold_per_length_peptides[allele_name][0][target_length])
                if "imm" not in allele_name: #do not subsambple for imm
                    peptides = set(np.random.choice(unique_peptides, size=PER_LENGTH_SIZE[target_length], replace=False))
                else:
                    peptides = set(unique_peptides)
                print(f"Selected/Subsampled {len(peptides)} peptides")
                per_allele_per_length_peptide_sets[allele_name] = peptides
        # get all intersections
        intersections = list()
        for i in range(len(selected_alleles)):
            for j in range(i + 1, len(selected_alleles)):
                set1 = per_allele_per_length_peptide_sets[selected_alleles[i]]
                set2 = per_allele_per_length_peptide_sets[selected_alleles[j]]
                intersect = set1.intersection(set2)
                print(f"{selected_alleles[i]} intersect {selected_alleles[j]} - {len(intersect)}")
                intersections.append(intersect)
        # delete all intersections
        for allele_name, peptide_set in per_allele_per_length_peptide_sets.items():
            print(f"Allele {allele_name}: deleted", end=" ")
            initial_length = len(peptide_set)
            for intersect in intersections:
                peptide_set = peptide_set.difference(intersect)
            result_length = len(peptide_set)
            print(f" {initial_length - result_length} of intersected peptides")
            per_allele_result_peptide_sets[allele_name] = per_allele_result_peptide_sets[allele_name].union(peptide_set)
    return per_allele_result_peptide_sets

print("Train")
per_allele_peptide_train_sets = handle_intersections(per_allele_per_kfold_per_length_binders_train)
print("Test")
per_allele_peptide_test_sets = handle_intersections(per_allele_per_kfold_per_length_binders_test)



train_peptides_b = set(set().union(*per_allele_peptide_train_sets.values()))

test_peptides_b = set(set().union(*per_allele_peptide_test_sets.values()))

print(f"Train {len(train_peptides_b)}, Test {len(test_peptides_b)}")



data_df = pd.DataFrame(index=list(train_peptides_b))
for allele_name in selected_alleles:
    data_df.loc[list(per_allele_peptide_train_sets[allele_name]), 'allele'] = allele_name



Train
Min unique sizes {12: 277, 13: 442, 14: 575, 15: 520, 16: 456, 17: 329, 18: 156}
Length 12
initial length for  allele HLA-DQA1*05:01/DQB1*02:01 TL 12 is 277
unique length for  allele HLA-DQA1*05:01/DQB1*02:01 TL 12 is 277
Selected/Subsampled 277 peptides
Allele HLA-DQA1*05:01/DQB1*02:01: deleted  0 of intersected peptides
Length 13
initial length for  allele HLA-DQA1*05:01/DQB1*02:01 TL 13 is 442
unique length for  allele HLA-DQA1*05:01/DQB1*02:01 TL 13 is 442
Selected/Subsampled 442 peptides
Allele HLA-DQA1*05:01/DQB1*02:01: deleted  0 of intersected peptides
Length 14
initial length for  allele HLA-DQA1*05:01/DQB1*02:01 TL 14 is 575
unique length for  allele HLA-DQA1*05:01/DQB1*02:01 TL 14 is 575
Selected/Subsampled 575 peptides
Allele HLA-DQA1*05:01/DQB1*02:01: deleted  0 of intersected peptides
Length 15
initial length for  allele HLA-DQA1*05:01/DQB1*02:01 TL 15 is 520
unique length for  allele HLA-DQA1*05:01/DQB1*02:01 TL 15 is 520
Selected/Subsampled 520 peptides
Allele HLA

In [259]:
from data_reading_methods import merge_per_allele_sets_into_combination_sets

per_allele_per_kfold_per_length_binders_train_old = per_allele_per_kfold_per_length_binders_train
per_allele_per_kfold_per_length_binders_test_old = per_allele_per_kfold_per_length_binders_test

per_allele_per_kfold_per_length_binders_train, per_allele_per_kfold_per_length_binders_test = merge_per_allele_sets_into_combination_sets(
    per_allele_per_kfold_per_length_binders_train, per_allele_per_kfold_per_length_binders_test,
    combination_size=combination_size,
    selected_alelles=ALLELES,
    total_splits=TOTAL_SPLITS
)
OLD_ALLELES = ALLELES
NEW_ALLELES = list(per_allele_per_kfold_per_length_binders_test.keys())
ALLELES = NEW_ALLELES

HLA-DQA1*05:01/DQB1*02:01
Length 8: HLA-DQA1*05:01/DQB1*02:01 - 13 
Length 9: HLA-DQA1*05:01/DQB1*02:01 - 91 
Length 10: HLA-DQA1*05:01/DQB1*02:01 - 87 
Length 11: HLA-DQA1*05:01/DQB1*02:01 - 152 
Length 12: HLA-DQA1*05:01/DQB1*02:01 - 277 
Length 13: HLA-DQA1*05:01/DQB1*02:01 - 442 
Length 14: HLA-DQA1*05:01/DQB1*02:01 - 575 
Length 15: HLA-DQA1*05:01/DQB1*02:01 - 520 
Length 16: HLA-DQA1*05:01/DQB1*02:01 - 456 
Length 17: HLA-DQA1*05:01/DQB1*02:01 - 329 
Length 18: HLA-DQA1*05:01/DQB1*02:01 - 156 
Length 19: HLA-DQA1*05:01/DQB1*02:01 - 107 
Length 20: HLA-DQA1*05:01/DQB1*02:01 - 75 
Length 21: HLA-DQA1*05:01/DQB1*02:01 - 55 
Length 22: HLA-DQA1*05:01/DQB1*02:01 - 34 
Length 23: HLA-DQA1*05:01/DQB1*02:01 - 26 
Length 24: HLA-DQA1*05:01/DQB1*02:01 - 18 
Length 25: HLA-DQA1*05:01/DQB1*02:01 - 14 


In [260]:
ALLELES

['HLA-DQA1*05:01/DQB1*02:01']

In [261]:
per_allele_peptide_train_sets.keys()

dict_keys(['HLA-DQA1*05:01/DQB1*02:01'])

In [262]:
selected_alleles

['HLA-DQA1*05:01/DQB1*02:01']

In [263]:
data_df.allele.value_counts()

HLA-DQA1*05:01/DQB1*02:01    2755
Name: allele, dtype: int64

# Read models

In [264]:
MODELS_DESCRIPTIONS = {
                    "simple_model_enrichmenet": {
        "path": "experiment_results/'N_successful'/",
        "states": 7,
        "params_string": "400-min_iters",
        "combination_size": 1,
        "alleles": [
                # 'HLA-DRB1*01:01'
                    'HLA-DPA1*02:02/DPB1*05:01',
                    #'HLA-A*03:01'
                    ],
        "target_lengths": [12, 13,14, 15, 16, 17, 18],
        "core_start_state": [7],
        "core_end_state": [8],
        "need_to_delete": [111],
        "skip_states": [],

    },

}
# add splited models
iterations = ["1111", "1112", "1121", "1122", "1211", "1212", "1221", "1222"]
for iteration in iterations:
    model_dict = {
        "path": f"experiment_results/'N_successful'/new_model{iteration}/",
        "states": 7,
        "params_string": "400-min_iters",
        "combination_size": 1,
        "alleles": [
                # 'HLA-DRB1*01:01'

               'HLA-DQA1*05:01/DQB1*02:01'
                 #   'HLA-DPA1*02:02/DPB1*05:01',
                    #'HLA-A*03:01'
                    ],
        "target_lengths": [12, 13,14, 15, 16, 17, 18],
        "core_start_state": [7],
        "core_end_state": [8],
        "need_to_delete": [111],
        "skip_states": [],
    }
    MODELS_DESCRIPTIONS[iteration] = model_dict

# iterations = ["1111", "1112", "1121", "1122", "1211", "1212", "1221", "1222"]
# for iteration in iterations:
#     model_dict = {
#         "path": f"experiment_results/'N_successful'/new_model{iteration}/",
#         "states": 14,
#         "params_string": "400-min_iters",
#         "combination_size": 1,
#         "alleles": [
#                 # 'HLA-DRB1*01:01'
#
#                'HLA-DQA1*05:01/DQB1*02:01'
#                  #   'HLA-DPA1*02:02/DPB1*05:01',
#                     #'HLA-A*03:01'
#                     ],
#         "target_lengths": [12, 13,14, 15, 16, 17, 18],
#         "core_start_state": [14],
#         "core_end_state": [15],
#         "need_to_delete": [111],
#         "skip_states": [],
#     }
#     MODELS_DESCRIPTIONS[iteration] = model_dict

In [265]:
selected_models = [
     "simple_model_enrichmenet"
] + iterations
selected_models = iterations


In [266]:
from hmm_logic_methods import load_model

In [267]:
import itertools

name_to_true_name = dict()
name_to_run = dict()
name_to_split = dict()
name_to_type = dict()
name_to_model_root = dict()
name_to_target_lengths = dict()
selected_binding_models = list()
per_name_binding_model = dict()


def populate_models_infos(model_name, run_num, split_num, model_path, model_type, target_lengths ):
    selected_binding_models.append(model_name)
    name_to_true_name[model_name] = model_name.split("#")[0]
    name_to_run[model_name] = run_num
    name_to_split[model_name] = split_num
    name_to_model_root[model_name] = model_path
    assert model_type in ["bw", "v", "bwv", "bw-it", "bwv-it", "v-it"] or "mixed" in model_type
    name_to_type[model_name] = model_type
    name_to_target_lengths[model_name] = target_lengths

for model in selected_models:
    model_description = MODELS_DESCRIPTIONS[model]
    combination_size = model_description['combination_size']
    model_alleles = model_description['alleles']
    MODELS_PATH = model_description['path']
    params_string = model_description['params_string']
    states = model_description['states']
    TARGET_LENGTHS = model_description['target_lengths']
    target_length_string = "-".join([str(l) for l in TARGET_LENGTHS])
    MIXED_ALLELES = list()

    model_alleles = set(model_alleles).intersection(set(OLD_ALLELES))
    for combination in itertools.combinations(sorted(model_alleles), combination_size):
        allele_name = "+".join(combination)
        # allele_name = f"{allele_name1}+{allele_name2}"
        MIXED_ALLELES.append(allele_name)

    for allele_name in MIXED_ALLELES:
        target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_').replace('/', '_')
        for run_num_b in range(RUNS):
            for split_num_b in range(TOTAL_SPLITS):
                #bw
                if 'bw' in models_to_read:
                    bw_model_name = f"{run_num_b}_run-{target_length_string}-{states}-st-baum-welch-alg-{params_string}_model-{target_allele_name}-{split_num_b}"
                    print(bw_model_name)
                    populate_models_infos(bw_model_name, run_num_b,split_num_b, MODELS_PATH, f"bw-it-mixed({target_allele_name})", TARGET_LENGTHS)
                if 'bwv' in models_to_read:
                    #bwv
                    bwv_model_name = f"{run_num_b}_run-{target_length_string}-{states}-st-baum-welch-alg-{params_string}_model-{target_allele_name}-{split_num_b}_plus_viterbi"
                    populate_models_infos(bwv_model_name, run_num_b,split_num_b, MODELS_PATH, f"bwv-it-mixed({target_allele_name})", TARGET_LENGTHS)
                if 'v' in models_to_read:
                    #v
                    v_model_name = f"{run_num_b}_run-{target_length_string}-{states}-st-viterbi-alg-{params_string}_model-{target_allele_name}-{split_num_b}"
                    v_model_name = v_model_name + "#" + model
                    populate_models_infos(v_model_name, run_num_b,split_num_b, MODELS_PATH, f"v-it-mixed({target_allele_name})", TARGET_LENGTHS)



for model_name in selected_binding_models:
    model = load_model(name_to_model_root[model_name], name_to_true_name[model_name])
    model.name = model_name
    per_name_binding_model[model_name] =  model

In [268]:
model_alleles = model_description['alleles']
MIXED_ALLELES = list()

model_alleles = set(model_alleles).intersection(set(OLD_ALLELES))
for combination in itertools.combinations(sorted(model_alleles), combination_size):
    allele_name = "+".join(combination)
    # allele_name = f"{allele_name1}+{allele_name2}"
    MIXED_ALLELES.append(allele_name)

In [269]:
model_description['alleles']

['HLA-DQA1*05:01/DQB1*02:01']

In [270]:
OLD_ALLELES

['HLA-DQA1*05:01/DQB1*02:01']

In [271]:
model_alleles

{'HLA-DQA1*05:01/DQB1*02:01'}

In [272]:
MODELS_DESCRIPTIONS[model]

KeyError: {
    "class" : "HiddenMarkovModel",
    "name" : "4_run-12-13-14-15-16-17-18-7-st-viterbi-alg-400-min_iters_model-HLA_DQA1_05_01_DQB1_02_01-0#1222",
    "start" : {
        "class" : "State",
        "distribution" : null,
        "name" : "grouped_model-start",
        "weight" : 1.0
    },
    "end" : {
        "class" : "State",
        "distribution" : null,
        "name" : "grouped_model-end",
        "weight" : 1.0
    },
    "states" : [
        {
            "class" : "State",
            "distribution" : {
                "class" : "Distribution",
                "dtype" : "str",
                "name" : "DiscreteDistribution",
                "parameters" : [
                    {
                        "A" : 0.022431817457239833,
                        "C" : 0.0021319492937060757,
                        "D" : 0.06829028969551708,
                        "E" : 0.05202129788815126,
                        "F" : 0.06908221326236522,
                        "G" : 

## Mark important states in model(will be used for core call)

In [273]:
import logomaker
from collections import defaultdict
def probs_to_info_matrix(distribution):
    amino_acids = list(distribution.parameters[0].keys())
    dummy_df = pd.DataFrame(columns = sorted(amino_acids), index=[0],  dtype=float)
    for i, (acid, prob) in enumerate(distribution.parameters[0].items()):
        dummy_df.loc[0, acid] = prob
    dummy_df.index.name = "pos"
    info_matrix = logomaker.transform_matrix(dummy_df, from_type='probability', to_type='information')
    return info_matrix

per_model_total_df = dict()

for model_key, model in per_name_binding_model.items():
    per_state_information = dict()
    per_state_distribution = dict()
    per_level_information = defaultdict(float)

    for state in model.states:
        if state not in [model.start, model.end]:
            info_matrix = probs_to_info_matrix(state.distribution)
            per_state_distribution[state.name] = state.distribution.parameters[0]
            per_state_information[state.name] = sum(info_matrix.loc[0])
            # per_level_information[name_to_level[state.name]] += per_state_information[state.name]
    info_df = pd.DataFrame(per_state_information.items(), columns=["state", "info"])
    distr_df = pd.DataFrame(per_state_distribution.values())
    total_df = pd.concat([info_df, distr_df], axis=1)
    per_model_total_df[model_key] = total_df

In [274]:
filtered_dfs = list()
per_model_important_states = dict()
per_model_skip_states = dict()

if models_to_read == ['bw']:
    for i, (model_key, total_df) in enumerate(per_model_total_df.items()):
        total_df = total_df.sort_values(by='info', ascending=False).copy()
        total_df = total_df[total_df['info'] >= 1]
        total_df['model'] = i
        total_df['model_name'] = model_key
        filtered_dfs.append(total_df)
        per_model_important_states[model_key] = list(total_df['state'].values)

if models_to_read == ['v']:
    per_model_anchors = dict()
    per_name_models = per_name_binding_model

    core_start_states= MODELS_DESCRIPTIONS[selected_models[0]]['core_start_state']
    core_end_states = MODELS_DESCRIPTIONS[selected_models[0]]['core_end_state']
    skip_states =  MODELS_DESCRIPTIONS[selected_models[0]]['skip_states']

    for i, (model_key, total_df) in enumerate(per_model_total_df.items()):
        total_df = total_df.sort_values(by='info', ascending=False).copy()
        total_df = total_df.loc[core_start_states + core_end_states]
        total_df['model'] = i
        total_df['model_name'] = model_key
        filtered_dfs.append(total_df)
        per_model_important_states[model_key] = list(total_df['state'].values)
    for i, (model_key, total_df) in enumerate(per_model_total_df.items()):
  
        total_df = total_df.sort_values(by='info', ascending=False).copy()
        total_df = total_df.loc[skip_states]
        total_df['model'] = i
        total_df['model_name'] = model_key
        filtered_dfs.append(total_df)
        per_model_skip_states[model_key] = list(total_df['state'].values)
        





In [239]:
data_df

,allele
NFELINHDVVEPLY,HLA-DQA1*05:01/DQB1*02:01
SPTSGGGKVAPAQPS,HLA-DQA1*05:01/DQB1*02:01
EEEVEFISVPVPEFA,HLA-DQA1*05:01/DQB1*02:01
SPGQEVFYSCEPGYDLR,HLA-DQA1*05:01/DQB1*02:01
EFMFDFDGDEIFH,HLA-DQA1*05:01/DQB1*02:01
...,...
TDTVTEVVVPFN,HLA-DQA1*05:01/DQB1*02:01
DPPLIALDKDAPLRF,HLA-DQA1*05:01/DQB1*02:01
DIVMTQSPLSLPVTP,HLA-DQA1*05:01/DQB1*02:01
LMQALPMGALPQ,HLA-DQA1*05:01/DQB1*02:01


In [240]:
ALLELES

['HLA-DQA1*05:01/DQB1*02:01']

In [241]:
data_df['allele'].map(lambda x : x in ALLELES[0])

NFELINHDVVEPLY       True
SPTSGGGKVAPAQPS      True
EEEVEFISVPVPEFA      True
SPGQEVFYSCEPGYDLR    True
EFMFDFDGDEIFH        True
                     ... 
TDTVTEVVVPFN         True
DPPLIALDKDAPLRF      True
DIVMTQSPLSLPVTP      True
LMQALPMGALPQ         True
HPQVTVYPAKTQPLQH     True
Name: allele, Length: 2755, dtype: bool

## Calculate core guesses

In [242]:
per_name_models.keys()

dict_keys(['0_run-12-13-14-15-16-17-18-7-st-viterbi-alg-400-min_iters_model-HLA_DQA1_05_01_DQB1_02_01-0#1111', '1_run-12-13-14-15-16-17-18-7-st-viterbi-alg-400-min_iters_model-HLA_DQA1_05_01_DQB1_02_01-0#1111', '2_run-12-13-14-15-16-17-18-7-st-viterbi-alg-400-min_iters_model-HLA_DQA1_05_01_DQB1_02_01-0#1111', '3_run-12-13-14-15-16-17-18-7-st-viterbi-alg-400-min_iters_model-HLA_DQA1_05_01_DQB1_02_01-0#1111', '4_run-12-13-14-15-16-17-18-7-st-viterbi-alg-400-min_iters_model-HLA_DQA1_05_01_DQB1_02_01-0#1111', '0_run-12-13-14-15-16-17-18-7-st-viterbi-alg-400-min_iters_model-HLA_DQA1_05_01_DQB1_02_01-0#1112', '1_run-12-13-14-15-16-17-18-7-st-viterbi-alg-400-min_iters_model-HLA_DQA1_05_01_DQB1_02_01-0#1112', '2_run-12-13-14-15-16-17-18-7-st-viterbi-alg-400-min_iters_model-HLA_DQA1_05_01_DQB1_02_01-0#1112', '3_run-12-13-14-15-16-17-18-7-st-viterbi-alg-400-min_iters_model-HLA_DQA1_05_01_DQB1_02_01-0#1112', '4_run-12-13-14-15-16-17-18-7-st-viterbi-alg-400-min_iters_model-HLA_DQA1_05_01_DQB1_02_0

In [215]:
PATH_TO_SAVE_RESULTS

'G:/My Drive/Work/ITMO/data/simple_model_enrichment/peptide_cores/'

In [216]:
total_df

,state,info,A,C,D,E,F,G,H,I,...,P,Q,R,S,T,V,W,Y,model,model_name


In [217]:
probs =  total_df[[f"prob_{i}" for i in range(RUNS*len(selected_models))]]

KeyError: "None of [Index(['prob_0', 'prob_1', 'prob_2', 'prob_3', 'prob_4', 'prob_5', 'prob_6',\n       'prob_7', 'prob_8', 'prob_9', 'prob_10', 'prob_11', 'prob_12',\n       'prob_13', 'prob_14', 'prob_15', 'prob_16', 'prob_17', 'prob_18',\n       'prob_19', 'prob_20', 'prob_21', 'prob_22', 'prob_23', 'prob_24',\n       'prob_25', 'prob_26', 'prob_27', 'prob_28', 'prob_29', 'prob_30',\n       'prob_31', 'prob_32', 'prob_33', 'prob_34', 'prob_35', 'prob_36',\n       'prob_37', 'prob_38', 'prob_39'],\n      dtype='object')] are in the [columns]"

In [275]:
# prepare peptides for viterbi calculation (expected to be unique)
from hmm_logic_methods import calculate_model_core_guesses
import os
import mhcgnomes
from scipy import stats


def get_mode_of_cluster_columns_start(row):
    columns = [f"core_start_{i}" for i in range(row['max_cluster']*RUNS, (row['max_cluster']+1)*RUNS)]
    return row[columns].mode()[0]
def get_mode_of_cluster_columns_end(row):
    columns = [f"core_end_{i}" for i in range(row['max_cluster']*RUNS, (row['max_cluster']+1)*RUNS)]
    return row[columns].mode()[0]


selected_peptides = []
for allele_name in ALLELES:
    print(allele_name)
    
    target_df = data_df[data_df['allele'].map(lambda x : x in allele_name)]
    allele_peptides = target_df.index.values
    total_df, selected_peptides_to_paths = calculate_model_core_guesses(allele_peptides, per_name_models, per_model_important_states, per_model_skip_states=per_model_skip_states,
                                    selected_peptides=selected_peptides
                                   )
    TOTAL_NUM_MODELS = RUNS*len(selected_models)
    # select best cluster for each peptide
    probs =  total_df[[f"prob_{i}" for i in range(TOTAL_NUM_MODELS)]]
    for i, cluster in enumerate(selected_models):
        cluster_probs = probs.iloc[:, i*RUNS:(i+1)*RUNS ]
        probs[f"cluster_{i}"] = cluster_probs.median(axis=1)
    max_cluster =  probs[[f"cluster_{i}" for i in range(len(selected_models))]].idxmax(axis=1).str.split("_").apply(lambda x: x[1]).astype(int)
    total_df['max_cluster'] = max_cluster
    
    # set core as most popular

    models_predictions_start = total_df[[f"core_start_{i}" for i in range(TOTAL_NUM_MODELS)] + ['max_cluster']]
    models_predictions_end = total_df[[f"core_end_{i}" for i in range(TOTAL_NUM_MODELS)] + ['max_cluster']]
    model_predictions_cores = total_df[[f"core_{i}" for i in range(TOTAL_NUM_MODELS)]]
    core_consensuses = model_predictions_cores.apply(lambda x: 
                    stats.mode(np.vstack(x.values))[0][0], axis=1)
    # core_starts = models_predictions_start.mode(axis=1)[0].astype(int)
    # core_ends = models_predictions_end.mode(axis=1)[0].astype(int)

    core_starts = total_df.apply(lambda row: get_mode_of_cluster_columns_start(row), axis=1)
    core_ends = total_df.apply(lambda row: get_mode_of_cluster_columns_end(row), axis=1)
    
    final_df = total_df.copy()
    selected_cluster_column = 'min_entropy_cluster'
    #selected_cluster_column = 'max_mean_prob_cluster'
    # final_df['core_start'] = 0
    # final_df['core_end'] = 0
    # for cluster in cluster_descriptions.index:
    #     index_peptides = final_df[final_df[selected_cluster_column] == cluster].index
    #     start_values = cluster_to_start_columns[cluster]
    #     end_values = cluster_to_end_columns[cluster]
    #     final_df.loc[index_peptides, 'core_start'] = start_values.loc[index_peptides].mean(axis=1).round().astype(int).values
    #     final_df.loc[index_peptides, 'core_start'] = start_values.loc[index_peptides].mode(axis=1)[0].astype(int)

    #     #final_df.loc[index_peptides, 'core_end'] = end_values.loc[index_peptides].mean(axis=1).round().astype(int).values
    #     final_df.loc[index_peptides, 'core_end'] = end_values.loc[index_peptides].mode(axis=1)[0].astype(int)

    final_df['core_start'] = core_starts
    final_df['core_end'] = core_ends

    final_df['core_peptide'] = final_df.apply(lambda row: row.name[row['core_start']:row['core_end'] + 1], axis=1)
    #final_df['core_peptide'] = final_df.apply(lambda row: "".join(np.array(list(row.name))[core_consensuses.loc[row.name].astype(bool)]), axis=1)

    final_df['core_len'] = final_df['core_peptide'].str.len()
    
    print(final_df['core_len'].value_counts())
    
    target_df['core_start'] = final_df['core_start']
    target_df['core_end'] = final_df['core_end']
    target_df['core_len'] = final_df['core_len']
    target_df['peptide_len'] = target_df.index.str.len()
    target_df['core_peptide'] = final_df['core_peptide']
    
    target_df = target_df[~target_df.index.duplicated(keep='first')]
    target_df['peptide']= target_df.index.values
    
    parsed_alleles = [mhcgnomes.parse(curr_name).compact_string(include_species=True).replace("*", "_") for curr_name in allele_name.split("+")]
    target_allele_name = "+".join(parsed_alleles)
    
    #arget_allele_name = mhcgnomes.parse(allele_name).compact_string(include_species=True).replace("*", "_")
    if not os.path.exists(PATH_TO_SAVE_RESULTS):
        os.makedirs(PATH_TO_SAVE_RESULTS)
    target_df.to_csv("{}cores#{}.csv".format(PATH_TO_SAVE_RESULTS,target_allele_name),sep=";", index=True)
    target_df.peptide.to_csv("{}peptides#{}.csv".format(PATH_TO_SAVE_RESULTS,target_allele_name),sep=";", index=False)






len(allele_peptides)

HLA-DQA1*05:01/DQB1*02:01
Model 0_run-12-13-14-15-16-17-18-7-st-viterbi-alg-400-min_iters_model-HLA_DQA1_05_01_DQB1_02_01-0#1111
Get viterbi paths for each peptide...


100%|██████████| 2755/2755 [00:00<00:00, 50091.93it/s]


Subselect unique paths (if unique path argument passed)..
	Unique paths: 2755 


100%|██████████| 2755/2755 [00:00<00:00, 28987.70it/s]


Model 1_run-12-13-14-15-16-17-18-7-st-viterbi-alg-400-min_iters_model-HLA_DQA1_05_01_DQB1_02_01-0#1111
Get viterbi paths for each peptide...


100%|██████████| 2755/2755 [00:00<00:00, 59284.13it/s]


... (154 outputs left out)

Subselect unique paths (if unique path argument passed)..
	Unique paths: 2755 


100%|██████████| 2755/2755 [00:00<00:00, 18345.22it/s]
C:\Projects\MHC_predictor\experiments\core_identification_simple_model_enrichment\hmm_logic_methods.py:1746: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  total_df[f'core_start_{i}'] =  viterbi_paths['simplified_path'].apply(lambda x: np.where(np.array(x) > 0)[0]).apply(lambda x: x[0] if len(x) > 0 else 0)
C:\Projects\MHC_predictor\experiments\core_identification_simple_model_enrichment\hmm_logic_methods.py:1747: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  total_df[f'core_

9    2755
Name: core_len, dtype: int64


2755

In [134]:
TEST_PDB_CORES = False
if TEST_PDB_CORES:
    df=pd.read_csv('test/binding_cores', sep='\t')
    df= df[~df['Antigen'].str.contains('X')]
    df = df[df['Allele'] == 'DRB1*01:01']

    allele_peptides = df.Antigen.values
    true_labels=df['Core (PDB)'].values
    netmhcpan = df['Predicted core.5'].values

In [135]:
from hmm_logic_methods import calculate_model_core_guesses
selected_peptides = [
    # "APEEILAEKSFNTVPY", 'EALTFSREHGPGVAEPE',
    #                  'YYSLLMPILTLTRAL',
    #                      'SYLTNMFATWSP',
    #                  'NPLIRHENRMVLAST',
    #                      'KPPKPVSKMRMATPLLM',
                         ]
total_df, selected_peptides_to_paths = calculate_model_core_guesses(allele_peptides, per_name_models, per_model_important_states, per_model_skip_states=per_model_skip_states,
                                    selected_peptides=selected_peptides
                                   )

Model 0_run-12-13-14-15-16-17-18-7-st-viterbi-alg-400-min_iters_model-HLA_DQA1_05_01_DQB1_02_01-0#1111
Get viterbi paths for each peptide...


100%|██████████| 2755/2755 [00:00<00:00, 76700.97it/s]


Subselect unique paths (if unique path argument passed)..
	Unique paths: 2755 


100%|██████████| 2755/2755 [00:00<00:00, 25739.78it/s]


Model 1_run-12-13-14-15-16-17-18-7-st-viterbi-alg-400-min_iters_model-HLA_DQA1_05_01_DQB1_02_01-0#1111
Get viterbi paths for each peptide...


100%|██████████| 2755/2755 [00:00<00:00, 82052.05it/s]


... (151 outputs left out)

Model 4_run-12-13-14-15-16-17-18-7-st-viterbi-alg-400-min_iters_model-HLA_DQA1_05_01_DQB1_02_01-0#1222
Get viterbi paths for each peptide...


100%|██████████| 2755/2755 [00:00<00:00, 55098.74it/s]


Subselect unique paths (if unique path argument passed)..
	Unique paths: 2755 


100%|██████████| 2755/2755 [00:00<00:00, 17747.12it/s]
C:\Projects\MHC_predictor\experiments\core_identification_simple_model_enrichment\hmm_logic_methods.py:1746: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  total_df[f'core_start_{i}'] =  viterbi_paths['simplified_path'].apply(lambda x: np.where(np.array(x) > 0)[0]).apply(lambda x: x[0] if len(x) > 0 else 0)
C:\Projects\MHC_predictor\experiments\core_identification_simple_model_enrichment\hmm_logic_methods.py:1747: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  total_df[f'core_

In [78]:
data_df = data_df.join(total_df)


In [79]:
if TEST_PDB_CORES:
    models_predictions_start = total_df[[f"core_start_{i}" for i in range(RUNS)]]
    models_predictions_end = total_df[[f"core_end_{i}" for i in range(RUNS)]]
    core_starts = models_predictions_start.mode(axis=1)[0].astype(int)
    core_ends = models_predictions_end.mode(axis=1)[0].astype(int)
    
    total_df['core_start'] = core_starts
    total_df['core_end'] = core_ends

    total_df['core_peptide'] = total_df.apply(lambda row: row.name[row['core_start']:row['core_end'] + 1], axis=1)
    total_df['core_len'] = total_df['core_peptide'].str.len()
    total_df['peptide_len'] = total_df.index.str.len()
    total_df['core_peptide'] = total_df['core_peptide']
    total_df = total_df[~total_df.index.duplicated(keep='first')]


    total_df.to_csv('test/predicted_cores_cores.csv',sep=";", index=True)

## Set core based on most popular value

In [80]:
from scipy import stats

models_predictions_start = total_df[[f"core_start_{i}" for i in range(RUNS)]]
models_predictions_end = total_df[[f"core_end_{i}" for i in range(RUNS)]]
model_predictions_cores = total_df[[f"core_{i}" for i in range(RUNS)]]
core_consensuses = model_predictions_cores.apply(lambda x: 
                stats.mode(np.vstack(x.values))[0][0], axis=1)


C:\Users\denkl\AppData\Local\Temp\ipykernel_9280\2287683951.py:7: FutureWarning: Unlike other reduction functions (e.g. `skew`, `kurtosis`), the default behavior of `mode` typically preserves the axis it acts along. In SciPy 1.11.0, this behavior will change: the default value of `keepdims` will become False, the `axis` over which the statistic is taken will be eliminated, and the value None will no longer be accepted. Set `keepdims` to True or False to avoid this warning.
  stats.mode(np.vstack(x.values))[0][0], axis=1)


In [81]:

core_consensuses

MPQVPVFTVEVKNYGVK     [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, ...
RTELETLQKQKK          [0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...
SQEKQVTVLELFR         [0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...
IIMDVYSYSEKE          [0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...
QALVEHVKEECD          [0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...
                                            ...                        
GATGAIMTTETTKTARAT    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, ...
STKTSAKLSSYT          [0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...
IASRLGLYSKTSQS        [0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...
ELSAWTEKQKEERQF       [0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...
SVEEKRNSKMLQK         [0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...
Length: 5640, dtype: object

In [82]:
len([item for item in core_consensuses if 0 in item])

5640

In [83]:
core_starts = models_predictions_start.mode(axis=1)[0].astype(int)
core_ends = models_predictions_end.mode(axis=1)[0].astype(int)


In [84]:
final_df = total_df

In [85]:
selected_cluster_column = 'min_entropy_cluster'
#selected_cluster_column = 'max_mean_prob_cluster'
# final_df['core_start'] = 0
# final_df['core_end'] = 0
# for cluster in cluster_descriptions.index:
#     index_peptides = final_df[final_df[selected_cluster_column] == cluster].index
#     start_values = cluster_to_start_columns[cluster]
#     end_values = cluster_to_end_columns[cluster]
#     final_df.loc[index_peptides, 'core_start'] = start_values.loc[index_peptides].mean(axis=1).round().astype(int).values
#     final_df.loc[index_peptides, 'core_start'] = start_values.loc[index_peptides].mode(axis=1)[0].astype(int)

#     #final_df.loc[index_peptides, 'core_end'] = end_values.loc[index_peptides].mean(axis=1).round().astype(int).values
#     final_df.loc[index_peptides, 'core_end'] = end_values.loc[index_peptides].mode(axis=1)[0].astype(int)

final_df['core_start'] = core_starts
final_df['core_end'] = core_ends

#final_df['core_peptide'] = final_df.apply(lambda row: row.name[row['core_start']:row['core_end'] + 1], axis=1)
final_df['core_peptide'] = final_df.apply(lambda row: "".join(np.array(list(row.name))[core_consensuses.loc[row.name].astype(bool)]), axis=1)

final_df['core_len'] = final_df['core_peptide'].str.len()

In [86]:
final_df['core_len'].value_counts()

9     4327
10    1207
11     103
12       3
Name: core_len, dtype: int64

In [87]:
data_df['core_start'] = final_df['core_start']
data_df['core_end'] = final_df['core_end']
data_df['core_len'] = final_df['core_len']
data_df['peptide_len'] = data_df.index.str.len()
data_df['core_peptide'] = final_df['core_peptide']
data_df = data_df[~data_df.index.duplicated(keep='first')]

C:\Users\denkl\AppData\Local\Temp\ipykernel_9280\1549749149.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data_df['core_start'] = final_df['core_start']
C:\Users\denkl\AppData\Local\Temp\ipykernel_9280\1549749149.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data_df['core_end'] = final_df['core_end']
C:\Users\denkl\AppData\Local\Temp\ipykernel_9280\1549749149.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Conside

In [88]:
data_df['core_start'].value_counts()

4    1624
5    1459
6     826
3     818
2     426
7     286
8      84
1      71
0      31
9      15
Name: core_start, dtype: int64

# Save core data to files

In [89]:
import os
if not os.path.exists(PATH_TO_SAVE_RESULTS):
    os.makedirs(PATH_TO_SAVE_RESULTS)
data_df.to_csv("{}extractd_cores.csv".format(PATH_TO_SAVE_RESULTS,),sep=";", index=True)
data_df.peptide.to_csv("{}original_peptides.csv".format(PATH_TO_SAVE_RESULTS,),sep=";", index=False)

# Group models based on predictions

We got all the start predictions from all models. Now we want to group these models. (since we expect that models have disagrement)

We want to cluster columns based on how similar are predictions for the whole training data



In [65]:
models_predictions = total_df[[f"core_start_{i}" for i in range(RUNS)]]

In [ ]:
from sklearn.cluster import FeatureAgglomeration
distance_threshold = 150

agglo = FeatureAgglomeration(n_clusters=None, metric='euclidean', linkage='ward', pooling_func=np.mean, distance_threshold=distance_threshold)
result_start_df = agglo.fit_transform(models_predictions)

In [78]:
clustered_features = pd.DataFrame({"cluster": agglo.labels_})

clustered_features['model'] = models_predictions.columns


unique_labels_counts = clustered_features.cluster.value_counts().sort_values(ascending=False)
unique_labels_counts

1    15
2    10
3     6
0     4
Name: cluster, dtype: int64

In [79]:
cluster_descriptions = clustered_features.groupby('cluster').agg(lambda x: " ".join(x.values))
cluster_descriptions['len'] = unique_labels_counts
cluster_descriptions.sort_values(by='len', ascending=False)

,model,len
cluster,,
1,core_start_1 core_start_3 core_start_10 core_s...,15
2,core_start_2 core_start_5 core_start_6 core_st...,10
3,core_start_9 core_start_11 core_start_23 core_...,6
0,core_start_0 core_start_4 core_start_17 core_s...,4


In [80]:
cluster_to_start_columns = {i : total_df[cluster_descriptions.loc[i]['model'].split(' ')] for i in cluster_descriptions.index}

In [81]:
model_to_cluster = dict(clustered_features['cluster'])

## Add mean probability for cluster for each peptide

In [34]:
from collections import defaultdict
cluster_to_prob_column_names = defaultdict(list)
cluster_to_end_column_names = defaultdict(list)

for cluster in cluster_descriptions.index:
    print(cluster)
    total_df[f'cluster_prob_{cluster}'] = 0
    
# calculate mean for predictions for given cluster of models to 
for i in range(RUNS):
    total_df[f'cluster_prob_{model_to_cluster[i]}'] += total_df[f'prob_{i}']
    cluster_to_prob_column_names[model_to_cluster[i]].append(f'prob_{i}')
    cluster_to_end_column_names[model_to_cluster[i]].append(f'core_end_{i}')

for cluster in cluster_descriptions.index:
    print(cluster)
    total_df[f'cluster_prob_{cluster}'] = total_df[f'cluster_prob_{cluster}'] / cluster_descriptions.loc[cluster]['len']
    
total_df['max_mean_prob_cluster'] =   total_df[[f'cluster_prob_{cluster}' for cluster in cluster_descriptions.index]].idxmax(axis=1).str.split('_').map(lambda x : int(x[-1]))

cluster_to_prob_columns = {i : total_df[cluster_to_prob_column_names[i]] for i in cluster_to_prob_column_names}
cluster_to_end_columns = {i : total_df[cluster_to_end_column_names[i]] for i in cluster_to_end_column_names}


0
1
2
3
0
1
2
3


In [35]:
cluster_to_start_columns[1].shape

(5640, 15)

## Add entropy value for cluster for each column

In [36]:
def mean_predictive_entropy(predictions):
    entropy = np.mean(-np.sum(np.exp(predictions)*predictions, axis=1)/predictions.shape[1])
    return entropy

def predictive_entropy(predictions):
    entropy = -np.sum(np.exp(predictions)*predictions, axis=1)/predictions.shape[1]
    return entropy

print(mean_predictive_entropy(cluster_to_prob_columns[0]))
print(mean_predictive_entropy(cluster_to_prob_columns[1]))
#print(mean_predictive_entropy(cluster_to_prob_columns[2]))
#print(mean_predictive_entropy(cluster_to_prob_columns[3]))

cluster_to_start_entropy_df = {i: predictive_entropy(cluster_to_prob_columns[i]) for i in cluster_to_prob_columns.keys()}

0.15289490292766392
0.15267749732179514


In [37]:
df_concat = pd.concat(cluster_to_start_entropy_df.values(), axis=1)
min_entropy_clusters = df_concat.idxmin(axis=1)
df_concat.columns = [f'cluster_entropy_{cluster}' for cluster in df_concat.columns ]
df_concat['min_entropy_cluster'] = min_entropy_clusters

In [38]:
final_df = pd.concat([total_df, df_concat], axis=1)